# Week 5b Hands-On Lab — Prompting and Evaluating a Video World Model

**ESP3201 · formative hands-on lab.** Generation needs a free-tier Colab **GPU (T4) runtime**. If no GPU is available, use the **precomputed rollout-bank** fallback — you still do the full evaluation on real frames.

A video-generation model is a **world model**: a learned simulator of how a scene evolves. You will vary **prompting**, **input controls** (seed, frames, guidance), and **evaluation criteria**, and measure how each affects the quality, consistency, and *usefulness* of the rollout.

> **Report only frames your run actually produced** (or the provided rollout-bank clips). No projected results.

**Attribution.** Evaluation dimensions follow **VBench** (Huang et al., 2023); generation follows the **diffusers** text-to-video tutorials. Lab code is original to ESP3201.

## Setup

In [ ]:
import sys, os

COURSE_REPO_URL = "https://github.com/bingquan/esp3201-public.git"

os.system('pip install -q numpy pillow')
try:
    import video_eval
except ModuleNotFoundError:
    cand = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
    if os.path.exists(os.path.join(cand, 'video_eval.py')):
        sys.path.insert(0, cand)
    elif COURSE_REPO_URL:
        os.system(f'git clone -q {COURSE_REPO_URL} course_repo')
        sys.path.insert(0, 'course_repo/labs/week05_embodied_system_critique/starter')
    else:
        raise ModuleNotFoundError('video_eval.py not found. On Colab set COURSE_REPO_URL.')
    import video_eval
from video_eval import (score_clip, synth_clip, load_frames_from_dir,
                        frames_from_diffusers)
print('video_eval loaded.')

## Warm-up — what the metrics mean (no model needed)

Three synthetic clips show how the objective metrics behave: a static clip, a smoothly moving object, and pure noise.

In [ ]:
for kind in ('static', 'moving', 'noise'):
    print(score_clip(synth_clip(kind), label=kind))

**Read it:** high `temporal_consistency` with near-zero `motion_magnitude` means *nothing is happening*; high `flicker` means *global incoherence*. A useful world-model rollout needs motion that is consistent, not frozen and not chaotic.

## Generate clips with a real video model (Colab GPU)

Pinned to **`cerspense/zeroscope_v2_576w`**, **verified on a free T4** (fp16 + CPU offload, measured ~5.8 GB peak; see `pinned_models.md`). Swap `MODEL_ID` and re-check memory if you like.

> **Time budget (important).** On a free T4 each clip takes **roughly 1–3 minutes** (much slower than the lab's reference GPU). The default below runs **3 experiments**, which is enough to see the controls matter; only add more if your session has time. Generate one clip first to gauge your runtime before launching the grid.

In [ ]:
# Run on a GPU runtime.
os.system('pip install -q diffusers transformers accelerate torch')
import torch
from diffusers import DiffusionPipeline

MODEL_ID = 'cerspense/zeroscope_v2_576w'   # VERIFIED on free T4 (~5.8 GB fp16+offload)
pipe = DiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
try:
    pipe.enable_vae_slicing()
except Exception:
    pass

def generate(prompt, seed=0, num_frames=24, num_inference_steps=25,
             guidance_scale=9.0, negative_prompt=None):
    g = torch.Generator(device='cuda').manual_seed(seed)
    out = pipe(prompt, num_frames=num_frames, num_inference_steps=num_inference_steps,
               guidance_scale=guidance_scale, negative_prompt=negative_prompt,
               height=320, width=576, generator=g)
    return frames_from_diffusers(out)   # zeroscope returns numpy frames; adapter handles it

print('peak VRAM after load (GB):', round(torch.cuda.max_memory_allocated()/1e9, 2))

### Controlled experiments (3 by default)

Each changes **one factor** so you can attribute the effect:

1. **terse** vs **detailed** prompt — does detail + motion verbs help?
2. **detailed + negative prompt** — does suppressing 'blurry, distorted, flickering' clean it up?

Once these run, the worksheet invites you to add a **seed** change and a **guidance** change *if time allows*.

In [ ]:
experiments = {
  'terse':         dict(prompt='a robot arm picking up a cube'),
  'detailed':      dict(prompt='a robotic arm slowly grasping a red cube on a table, smooth motion, realistic'),
  'detailed+neg':  dict(prompt='a robotic arm slowly grasping a red cube on a table, smooth motion, realistic',
                        negative_prompt='blurry, distorted, flickering'),
}
scores = []
for label, kw in experiments.items():
    frames = generate(**kw)
    scores.append(score_clip(frames, label=label))
    print(scores[-1])

### Optional extra controls (only if your session has time)

```python
# same prompt, different seed -> how stable is the model?
print(score_clip(generate('a robot arm picking up a cube', seed=1), label='seedB'))
# higher guidance -> tighter prompt adherence, but can over-sharpen / flicker
print(score_clip(generate('a robot arm picking up a cube', guidance_scale=14.0), label='high_guidance'))
```

## No GPU? Use the precomputed rollout bank

Evaluate real generated frames committed under `rollout_bank/` without generating them yourself.

In [ ]:
import glob
base = next((p for p in ['../rollout_bank',
            'course_repo/labs/week05_embodied_system_critique/rollout_bank']
            if glob.glob(p + '/clip*')), None)
for d in sorted(glob.glob((base or '../rollout_bank') + '/clip*')):
    print(score_clip(load_frames_from_dir(d), label=d.split('/')[-1]))

## Worksheet (your deliverable)

### 1. Controls x criteria grid

Record objective metrics + your 1-5 human judgments for each experiment you ran:

| Experiment | Temporal consistency | Motion | Flicker | Prompt adherence (1-5) | Identity stability (1-5) | Physical plausibility (1-5) |
|------------|----------------------|--------|---------|------------------------|--------------------------|-----------------------------|
| terse | | | | | | |
| detailed | | | | | | |
| detailed+neg | | | | | | |
| *(optional)* seedB | | | | | | |
| *(optional)* high_guidance | | | | | | |

### 2. Which controls mattered?

- Which single control most improved a criterion you care about, with the numbers/judgments that show it?
- Where did the metrics and your eyes **disagree** (e.g. high consistency but the object morphed)? What does that say about trusting any single metric?

### 3. World-model-as-predictor

- At what `num_frames` / horizon did coherence break for your model?
- `Would you trust this model's rollout inside a planning loop? For what horizon, and why?`

## AI-Agent Usage Disclosure

State:

- which tools you used
- what they helped produce
- what you verified or rewrote yourself
- one specific thing you did not trust without checking